# Day 41 — Number theory for coders: fast power, GCD, modular inverse and the sieve

Take `% m` after every `+`, `-` and `*` and the final remainder is exactly the one the
full-size number would have had. That deal is why `pow(3, 10**18, 10**9 + 7)` needs only
82 multiplications and answers in microseconds.

The deal has one hole - **division** - and none of the ways it breaks raise an error:

* `fact[n] // (fact[k] * fact[n-k])` on factorials already reduced mod 1e9+7 is right up
  to n = 12 and wrong from 13 on;
* the "inverse" `pow(a, m - 2, m)` (Fermat) only works when m is prime;
* LeetCode 204 counts primes *below* n - counting up to n passes all three examples.

In [1]:
from __future__ import annotations
import math, random, sys, time
from collections import deque
from math import comb, isqrt
if hasattr(sys, 'set_int_max_str_digits'):
    sys.set_int_max_str_digits(0)

MOD = 10**9 + 7                 # prime - the modulus almost every judge uses


def best_time(fn, repeat: int = 3) -> float:
    best = float('inf')
    for _ in range(repeat):
        t = time.perf_counter()
        fn()
        best = min(best, time.perf_counter() - t)
    return best

## 1. What survives `%`

Addition, subtraction and multiplication can be reduced at every step. Division cannot:
`12 // 4 = 3`, but after reducing mod 5 the same division is `2 // 4 = 0`. Python's `%`
takes the sign of `m`, so `-7 % 3` is 2 (C and Java give -1).

In [2]:
a, b, m = 123_456_789, 987_654_321, 1_000
print((a + b) % m, (a % m + b % m) % m)
print((a - b) % m, (a % m - b % m) % m)
print((a * b) % m, (a % m) * (b % m) % m)
print('division:', (12 // 4) % 5, 'vs', (12 % 5) // (4 % 5))
print('sign:', -7 % 3, 'vs C-style', int(math.fmod(-7, 3)))

110 110
468 468
269 269
division: 3 vs 0
sign: 2 vs C-style -1


## 2. Fast power: square-and-multiply

Write the exponent in binary. Square once per bit; multiply into the result only on the
1 bits. 13 = 0b1101 = 8 + 4 + 1, so 3^13 = 3^8 · 3^4 · 3^1 - five multiplications
instead of twelve, and 82 instead of a billion billion for e = 10^18.

In [3]:
# --------------------------------------------------------------- 2. fast power
def power_mod(a: int, e: int, m: int) -> int:
    """Square-and-multiply: walk the bits of e from the lowest one upward."""
    result = 1 % m
    a %= m
    while e:
        if e & 1:                       # this bit is on: multiply its power in
            result = result * a % m
        a = a * a % m                   # a, a^2, a^4, a^8 ... one squaring per bit
        e >>= 1
    return result


def power_trace(a: int, e: int, m: int) -> list[dict]:
    """One row per bit of e: the bit, the current square, the running result."""
    rows, result, sq, k = [], 1 % m, a % m, 0
    while e:
        bit = e & 1
        if bit:
            result = result * sq % m
        rows.append({'k': k, 'bit': bit, 'square': sq, 'result': result})
        sq = sq * sq % m
        e >>= 1
        k += 1
    return rows


def multiplications(e: int) -> tuple[int, int]:
    """(naive loop, square-and-multiply) multiplication counts for a^e."""
    return e - 1 if e else 0, max(e.bit_length() - 1, 0) + max(e.bit_count() - 1, 0)


for r in power_trace(3, 13, 100):
    print(r)
print('3**13 % 100 =', 3**13 % 100)
for e in (13, 10**9, 10**18):
    print(f'e = {e:,}: naive / fast multiplications = {multiplications(e)}')
for e in (10**5, 10**6):
    t_big = best_time(lambda: 3**e % MOD)
    t_pow = best_time(lambda: pow(3, e, MOD))
    print(f'e = {e:,}: 3**e % MOD {t_big*1000:.2f} ms, pow(3, e, MOD) {t_pow*1e6:.1f} us')

{'k': 0, 'bit': 1, 'square': 3, 'result': 3}
{'k': 1, 'bit': 0, 'square': 9, 'result': 3}
{'k': 2, 'bit': 1, 'square': 81, 'result': 43}
{'k': 3, 'bit': 1, 'square': 61, 'result': 23}
3**13 % 100 = 23
e = 13: naive / fast multiplications = (12, 5)
e = 1,000,000,000: naive / fast multiplications = (999999999, 41)
e = 1,000,000,000,000,000,000: naive / fast multiplications = (999999999999999999, 82)
e = 100,000: 3**e % MOD 2.02 ms, pow(3, e, MOD) 1.8 us
e = 1,000,000: 3**e % MOD 71.87 ms, pow(3, e, MOD) 2.1 us


### LeetCode 50 - Pow(x, n), and Fibonacci by matrix power

Same loop on a float. A negative `n` becomes `1/x` with `-n`; Python ints mean
`-(-2**31)` does not overflow. The same squaring works on a 2x2 matrix: `[[1,1],[1,0]]^n`
holds F(n), so F(10^18) mod p needs 83 matrix multiplications.

In [4]:
class Solution50:
    """LeetCode 50 - Pow(x, n)."""

    def myPow(self, x: float, n: int) -> float:
        if n < 0:
            x, n = 1 / x, -n            # in Python -n never overflows
        result = 1.0
        while n:
            if n & 1:
                result *= x
            x *= x
            n >>= 1
        return result


def mat_mul(A, B, m):
    return [[(A[0][0] * B[0][0] + A[0][1] * B[1][0]) % m,
             (A[0][0] * B[0][1] + A[0][1] * B[1][1]) % m],
            [(A[1][0] * B[0][0] + A[1][1] * B[1][0]) % m,
             (A[1][0] * B[0][1] + A[1][1] * B[1][1]) % m]]


def fib_mod(n: int, m: int = MOD) -> tuple[int, int]:
    """F(n) mod m from [[1,1],[1,0]]^n - same square-and-multiply, on matrices.
    Returns (F(n) mod m, number of 2x2 matrix multiplications)."""
    R, A, mults = [[1, 0], [0, 1]], [[1, 1], [1, 0]], 0
    while n:
        if n & 1:
            R = mat_mul(R, A, m)
            mults += 1
        n >>= 1
        if n:
            A = mat_mul(A, A, m)
            mults += 1
    return R[0][1], mults


def fib_loop(n: int, m: int = MOD) -> int:
    a, b = 0, 1
    for _ in range(n):
        a, b = b, (a + b) % m
    return a


s = Solution50()
print(s.myPow(2.0, 10), round(s.myPow(2.1, 3), 5), s.myPow(2.0, -2))
print('n = -2**31: naive', f'{2**31:,}', 'multiplications, fast', multiplications(2**31)[1])
print('F(10**18) mod 1e9+7 and matrix multiplications:', fib_mod(10**18))

1024.0 9.261 0.25
n = -2**31: naive 2,147,483,648 multiplications, fast 31
F(10**18) mod 1e9+7 and matrix multiplications: (209783453, 83)


## 3. GCD: Euclid and extended Euclid

`gcd(a, b) = gcd(b, a % b)`: whatever divides a and b divides the remainder too. The
slowest input is a pair of consecutive Fibonacci numbers (every quotient is 1). Extended
Euclid keeps `a == A*x + B*y` on every row and so returns the coefficients as well.

In [5]:
# --------------------------------------------------------------------- 3. GCD
def gcd(a: int, b: int) -> int:
    """Euclid: gcd(a, b) = gcd(b, a % b), until the remainder is 0."""
    while b:
        a, b = b, a % b
    return abs(a)


def gcd_trace(a: int, b: int) -> list[tuple[int, int, int, int]]:
    """Rows (a, b, q, r) with a = q*b + r."""
    rows = []
    while b:
        q, r = divmod(a, b)
        rows.append((a, b, q, r))
        a, b = b, r
    return rows


def gcd_subtract_steps(a: int, b: int) -> int:
    """How many subtractions the 'subtract the smaller from the larger' gcd needs.
    Each Euclid step a % b replaces a // b subtractions, so the total is the sum
    of the quotients - computed here without actually doing them all."""
    steps = 0
    while b:
        steps += a // b
        a, b = b, a % b
    return steps - 1                     # the last quotient's final subtraction hits 0


def ext_gcd(a: int, b: int) -> tuple[int, int, int]:
    """(g, x, y) with a*x + b*y == g == gcd(a, b)."""
    x0, y0, x1, y1 = 1, 0, 0, 1
    while b:
        q, a, b = a // b, b, a % b
        x0, x1 = x1, x0 - q * x1
        y0, y1 = y1, y0 - q * y1
    return a, x0, y0


def ext_gcd_trace(a: int, b: int) -> list[tuple[int, int, int, int, int]]:
    """Rows (a, b, q, x, y): after each step a == a0*x + b0*y still holds."""
    a0, b0 = a, b
    x0, y0, x1, y1 = 1, 0, 0, 1
    rows = [(a, b, None, x0, y0)]
    while b:
        q, a, b = a // b, b, a % b
        x0, x1 = x1, x0 - q * x1
        y0, y1 = y1, y0 - q * y1
        rows.append((a, b, q, x0, y0))
        assert a == a0 * x0 + b0 * y0
    return rows


for row in gcd_trace(252, 105):
    print('%d = %d x %d + %d' % (row[0], row[2], row[1], row[3]))
print('subtraction steps, gcd(10**6, 1):', f'{gcd_subtract_steps(10**6, 1):,}', 'vs Euclid', len(gcd_trace(10**6, 1)))
print('consecutive Fibonacci (832040, 514229):', len(gcd_trace(832040, 514229)), 'steps')
for row in ext_gcd_trace(240, 46):
    print(row)
print(ext_gcd(240, 46))

252 = 2 x 105 + 42
105 = 2 x 42 + 21
42 = 2 x 21 + 0
subtraction steps, gcd(10**6, 1): 999,999 vs Euclid 1
consecutive Fibonacci (832040, 514229): 28 steps
(240, 46, None, 1, 0)
(46, 10, 5, 0, 1)
(10, 6, 4, 1, -5)
(6, 4, 1, -4, 21)
(4, 2, 1, 5, -26)
(2, 0, 2, -9, 47)
(2, -9, 47)


### LeetCode 365 - Water and Jug Problem

Every amount you can measure is a combination `x*i + y*j`, and Bezout says those are
exactly the multiples of `gcd(x, y)` - as long as it fits in both jugs together. The BFS
over (jug1, jug2) states is the brute-force check.

In [6]:
class Solution365:
    """LeetCode 365 - Water and Jug Problem."""

    def canMeasureWater(self, x: int, y: int, target: int) -> bool:
        return target <= x + y and target % math.gcd(x, y) == 0


def can_measure_bfs(x: int, y: int, target: int) -> bool:
    """Brute force over (jug1, jug2) states - the check for Solution365."""
    seen, q = {(0, 0)}, deque([(0, 0)])
    while q:
        a, b = q.popleft()
        if a + b == target:
            return True
        pour_ab, pour_ba = min(a, y - b), min(b, x - a)
        for s in ((x, b), (a, y), (0, b), (a, 0),
                  (a - pour_ab, b + pour_ab), (a + pour_ba, b - pour_ba)):
            if s not in seen:
                seen.add(s)
                q.append(s)
    return False


s = Solution365()
for args in ((3, 5, 4), (2, 6, 5), (1, 2, 3), (3, 5, 9)):
    print(args, s.canMeasureWater(*args), can_measure_bfs(*args))

(3, 5, 4) True True
(2, 6, 5) False False
(1, 2, 3) True True
(3, 5, 9) False False


## 4. Modular inverse - and Fermat on a composite modulus

`inverse(a, m)` comes from extended Euclid and exists only when `gcd(a, m) == 1`.
Fermat's shortcut `pow(a, m - 2, m)` is the same thing *only when m is prime*; on m = 10
it says the inverse of 3 is 1. `pow(a, -1, m)` (Python 3.8+) refuses instead of lying.

In [7]:
# ---------------------------------------------------------- 4. modular inverse
def inverse(a: int, m: int) -> int:
    """x with a*x % m == 1, via extended Euclid; exists only if gcd(a, m) == 1."""
    g, x, _ = ext_gcd(a % m, m)
    if g != 1:
        raise ValueError(f'{a} has no inverse mod {m}')
    return x % m


def inverse_fermat(a: int, m: int) -> int:
    """a^(m-2) mod m.  Correct only when m is prime - and nothing checks that."""
    return pow(a, m - 2, m)


def fermat_report(m: int) -> tuple[int, int]:
    """(invertible a in 1..m-1, how many of them Fermat gets wrong)."""
    inv = [a for a in range(1, m) if math.gcd(a, m) == 1]
    return len(inv), sum(a * inverse_fermat(a, m) % m != 1 for a in inv)


print(inverse(3, 10), pow(3, -1, 10), inverse_fermat(3, 10))
print(inverse(3, 11), inverse_fermat(3, 11))
for m in (10, 12, 1337, 1_000_000):
    print(m, 'invertible / Fermat wrong:', fermat_report(m))
try:
    pow(4, -1, 10)
except ValueError as err:
    print('pow(4, -1, 10):', err, '| pow(4, 8, 10) =', pow(4, 8, 10))

7 7 1
4 4
10 invertible / Fermat wrong: (4, 3)
12 invertible / Fermat wrong: (4, 3)
1337 invertible / Fermat wrong: (1140, 1136)
1000000 invertible / Fermat wrong: (400000, 399999)
pow(4, -1, 10): base is not invertible for the given modulus | pow(4, 8, 10) = 6


## 5. nCk mod p - the `//` that works until n = 13

12! = 479,001,600 is below 1e9+7 and 13! is above it, so from 13 on `fact[n]` is a
remainder and dividing remainders means nothing. Multiply by inverse factorials instead.

In [8]:
# ------------------------------------------------------------- 5. nCk mod p
def factorials(n: int, p: int = MOD) -> list[int]:
    f = [1] * (n + 1)
    for i in range(1, n + 1):
        f[i] = f[i - 1] * i % p
    return f


def ncr_floor_div(n: int, k: int, f: list[int], p: int = MOD) -> int:
    """The bug: integer division on numbers that have already been reduced mod p."""
    return f[n] // (f[k] * f[n - k] % p) % p


class Binomial:
    """nCk mod p for many queries: O(n) table once, O(1) per query."""

    def __init__(self, n: int, p: int = MOD):
        self.p = p
        self.f = factorials(n, p)
        self.inv_f = [1] * (n + 1)
        self.inv_f[n] = pow(self.f[n], p - 2, p)       # p is prime: Fermat is fine here
        for i in range(n, 0, -1):                      # 1/(i-1)! = i * 1/i!
            self.inv_f[i - 1] = self.inv_f[i] * i % p

    def ncr(self, n: int, k: int) -> int:
        if k < 0 or k > n:
            return 0
        return self.f[n] * self.inv_f[k] % self.p * self.inv_f[n - k] % self.p


f = factorials(200)
print([ (n, ncr_floor_div(n, 2, f), comb(n, 2)) for n in range(11, 16)])
bad = sum(ncr_floor_div(n, k, f) != comb(n, k) % MOD for n in range(31) for k in range(n + 1))
print('n <= 30 wrong:', bad, 'of', sum(n + 1 for n in range(31)))
B = Binomial(10**6)
print(B.ncr(100, 50), B.ncr(10**6, 500_000))

[(11, 55, 55), (12, 66, 66), (13, 2, 78), (14, 0, 91), (15, 1, 105)]
n <= 30 wrong: 369 of 496
538992043 996692777


## 6. Sieve of Eratosthenes - LeetCode 204

Each prime p crosses out its multiples starting at p*p (smaller ones were crossed by a
smaller prime), and the outer loop stops at sqrt(n). Assigning a slice of a bytearray
does the inner loop in C.

### LeetCode 204 - Count Primes
Count the primes strictly **below** n. Counting up to n passes all three examples
(10, 0, 1) and is wrong exactly when n is prime.

In [9]:
# ------------------------------------------------------------------ 6. sieve
def is_prime_trial(n: int) -> bool:
    if n < 2:
        return False
    for d in range(2, isqrt(n) + 1):
        if n % d == 0:
            return False
    return True


def sieve(n: int) -> bytearray:
    """is_prime[i] for 0 <= i < n.  Cross out multiples of each prime p,
    starting at p*p (smaller multiples were crossed by smaller primes),
    and only for p*p < n (a composite below n has a factor <= sqrt(n))."""
    is_prime = bytearray([1]) * max(n, 2)
    is_prime[0] = is_prime[1] = 0
    for p in range(2, isqrt(n - 1) + 1 if n > 1 else 2):
        if is_prime[p]:
            is_prime[p * p::p] = bytes(len(range(p * p, n, p)))
    return is_prime[:n]


def sieve_loop(n: int, start_at_square: bool = True, stop_at_root: bool = True):
    """The same sieve as plain loops, counting every crossing-out it does."""
    is_prime = [True] * max(n, 2)
    is_prime[0] = is_prime[1] = False
    crossings = 0
    top = isqrt(n - 1) if stop_at_root else n - 1
    for p in range(2, top + 1):
        if is_prime[p]:
            for q in range(p * p if start_at_square else 2 * p, n, p):
                is_prime[q] = False
                crossings += 1
    return is_prime[:n], crossings


def sieve_trace(n: int) -> list[tuple[int, list[int]]]:
    """(p, the numbers p crosses out for the first time) for each sieving prime."""
    is_prime = [True] * n
    is_prime[0] = is_prime[1] = False
    out = []
    for p in range(2, isqrt(n - 1) + 1):
        if is_prime[p]:
            new = [q for q in range(p * p, n, p) if is_prime[q]]
            for q in range(p * p, n, p):
                is_prime[q] = False
            out.append((p, new))
    return out


class Solution204:
    """LeetCode 204 - Count Primes (strictly less than n)."""

    def countPrimes(self, n: int) -> int:
        return sum(sieve(n)) if n > 2 else 0


def count_primes_upto(n: int) -> int:
    """The bug: primes <= n instead of < n.  Passes n = 10, 0, 1."""
    return sum(sieve(n + 1)) if n >= 2 else 0


for p, new in sieve_trace(50):
    print(p, new)
print('crossings n = 10**6:', sieve_loop(10**6)[1], sieve_loop(10**6, start_at_square=False)[1])
t0 = time.perf_counter(); c = sum(sieve(10**6)); t1 = time.perf_counter()
print(f'primes below 10**6: {c:,} in {(t1 - t0)*1000:.1f} ms')
s = Solution204()
print([(n, s.countPrimes(n), count_primes_upto(n)) for n in (10, 0, 1, 13)])
print('n = 0..5000 wrong:', sum(s.countPrimes(n) != count_primes_upto(n) for n in range(5001)))

2 [4, 6, 8, 10, 12, 14, 16, 18, 20, 22, 24, 26, 28, 30, 32, 34, 36, 38, 40, 42, 44, 46, 48]
3 [9, 15, 21, 27, 33, 39, 45]
5 [25, 35]
7 [49]
crossings n = 10**6: 2122046 2197837
primes below 10**6: 78,498 in 16.5 ms
[(10, 4, 4), (0, 0, 0), (1, 0, 0), (13, 5, 6)]
n = 0..5000 wrong: 669


## 7. LeetCode 1922 and LeetCode 372

1922: even positions have 5 choices (0 2 4 6 8), odd positions 4 (2 3 5 7) - the answer
is `5^ceil(n/2) * 4^floor(n/2) mod p` for n up to 10^15, which only `pow` can do.

372: the exponent is a digit list. Process it digit by digit with
`result = result^10 * a^d`. Reducing the exponent mod 1336 assumes 1337 is prime; it is
7 x 191, and that shortcut still passes all four examples.

In [10]:
# --------------------------------------------------------------- 7. LC 1922 / 372
class Solution1922:
    """LeetCode 1922 - Count Good Numbers."""

    def countGoodNumbers(self, n: int) -> int:
        even_slots, odd_slots = (n + 1) // 2, n // 2   # indices 0, 2, 4 ... / 1, 3, 5 ...
        return pow(5, even_slots, MOD) * pow(4, odd_slots, MOD) % MOD


class Solution372:
    """LeetCode 372 - Super Pow: a^b mod 1337, b given as a list of digits."""

    def superPow(self, a: int, b: list[int]) -> int:
        result = 1
        for d in b:                      # a^(10*e + d) = (a^e)^10 * a^d
            result = pow(result, 10, 1337) * pow(a, d, 1337) % 1337
        return result


def super_pow_fermat(a: int, b: list[int]) -> int:
    """The bug: shrink the exponent mod 1336 as if 1337 were prime (1337 = 7 * 191)."""
    e = int(''.join(map(str, b)))
    return pow(a, e % 1336, 1337)


s = Solution1922()
print([s.countGoodNumbers(n) for n in (1, 4, 50, 10**15)])
s = Solution372()
for a, b in ((2, [3]), (2, [1, 0]), (1, [4, 3, 3, 8, 5, 2]), (2147483647, [2, 0, 0])):
    print(a, b, s.superPow(a, b), super_pow_fermat(a, b))

[5, 400, 564908303, 711414395]
2 [3] 8 8
2 [1, 0] 1024 1024
1 [4, 3, 3, 8, 5, 2] 1 1
2147483647 [2, 0, 0] 1198 1198


## Tests

In [11]:
def run_tests() -> None:
    assert (-7) % 3 == 2 and 12 // 4 % 5 == 3 and (12 % 5) // (4 % 5) == 0
    rng = random.Random(1)
    for _ in range(500):
        a, e, m = rng.randrange(10**9), rng.randrange(10**6), rng.randrange(1, 10**9)
        assert power_mod(a, e, m) == pow(a, e, m)
    assert power_mod(5, 0, 1) == 0 and power_trace(3, 13, 100)[-1]['result'] == 23
    assert multiplications(13) == (12, 5)
    s = Solution50()
    assert s.myPow(2.0, 10) == 1024.0 and abs(s.myPow(2.1, 3) - 9.261) < 1e-9 and s.myPow(2.0, -2) == 0.25
    assert all(fib_mod(n)[0] == fib_loop(n) for n in range(300))
    for _ in range(500):
        a, b = rng.randrange(10**12), rng.randrange(10**12)
        g, x, y = ext_gcd(a, b)
        assert g == math.gcd(a, b) == gcd(a, b) and a * x + b * y == g
    assert gcd(252, 105) == 21 and len(gcd_trace(832040, 514229)) == 28
    for a in range(1, 60):
        for b in range(1, 60):
            x, y, steps = a, b, 0
            while x != y:
                x, y = (x - y, y) if x > y else (x, y - x)
                steps += 1
            assert gcd_subtract_steps(max(a, b), min(a, b)) == steps
    s = Solution365()
    for x in range(1, 9):
        for y in range(1, 9):
            for t in range(0, 20):
                assert s.canMeasureWater(x, y, t) == can_measure_bfs(x, y, t)
    assert inverse(3, 10) == 7 and inverse_fermat(3, 10) == 1
    for m in (7, 11, 97, 101):
        assert all(inverse_fermat(a, m) == inverse(a, m) == pow(a, -1, m) for a in range(1, m))
    assert fermat_report(10) == (4, 3)
    f = factorials(60)
    assert all(ncr_floor_div(n, k, f) == comb(n, k) for n in range(13) for k in range(n + 1))
    assert ncr_floor_div(13, 1, f) != 13
    B = Binomial(500)
    assert all(B.ncr(n, k) == comb(n, k) % MOD for n in range(200) for k in range(n + 1))
    assert sum(sieve(10**6)) == 78498 and sum(sieve(100)) == 25
    assert all(bool(v) == is_prime_trial(i) for i, v in enumerate(sieve(5000)))
    assert sieve_loop(1000)[0] == [bool(v) for v in sieve(1000)]
    s = Solution204()
    assert [s.countPrimes(n) for n in (10, 0, 1)] == [4, 0, 0]
    assert [count_primes_upto(n) for n in (10, 0, 1)] == [4, 0, 0] and count_primes_upto(3) == 2
    s = Solution1922()
    assert [s.countGoodNumbers(n) for n in (1, 4, 50)] == [5, 400, 564908303]
    s = Solution372()
    assert s.superPow(2, [3]) == 8 and s.superPow(2, [1, 0]) == 1024
    assert s.superPow(1, [4, 3, 3, 8, 5, 2]) == 1 and s.superPow(2147483647, [2, 0, 0]) == 1198
    for _ in range(300):
        a = rng.randrange(1, 2**31)
        b = [rng.randrange(1, 10)] + [rng.randrange(10) for _ in range(rng.randrange(0, 8))]
        assert s.superPow(a, b) == pow(a, int(''.join(map(str, b))), 1337)


run_tests()
print('all assertions passed')

all assertions passed
